In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
from google.colab import files

uploaded = files.upload()


Saving WA_Fn-UseC_-Telco-Customer-Churn.csv to WA_Fn-UseC_-Telco-Customer-Churn (1).csv


In [8]:
df = pd.read_csv("WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [9]:
print("BEFORE PREPROCESSING")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

BEFORE PREPROCESSING
Rows: 7043
Columns: 21


In [10]:
print("Column Names:")
print(df.columns.tolist())

Column Names:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [12]:
print("Missing values in each column:")
print(df.isnull().sum())

print("\nTotal missing values:", df.isnull().sum().sum())

Missing values in each column:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

Total missing values: 0


In [13]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [14]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
SeniorCitizen,7043.0,0.162147,0.368612,0.00,0.0,0.00,0.00,1.00
tenure,7043.0,32.371149,24.559481,0.00,9.0,29.00,55.00,72.00
MonthlyCharges,7043.0,64.761692,30.090047,18.25,35.5,70.35,89.85,118.75


In [15]:
print("Blank TotalCharges values:",
      (df["TotalCharges"].str.strip() == "").sum())

Blank TotalCharges values: 11


In [16]:
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

In [17]:
print("Missing TotalCharges after conversion:",
      df["TotalCharges"].isnull().sum())

Missing TotalCharges after conversion: 11


In [18]:
median_total_charges = df["TotalCharges"].median()

df["TotalCharges"] = df["TotalCharges"].fillna(
    median_total_charges
)

print("Missing TotalCharges after filling:",
      df["TotalCharges"].isnull().sum())

Missing TotalCharges after filling: 0


In [19]:
print("TotalCharges data type:", df["TotalCharges"].dtype)
print("Total missing values:", df.isnull().sum().sum())

TotalCharges data type: float64
Total missing values: 0


In [20]:
df["AvgMonthlySpend"] = df["TotalCharges"] / df["tenure"].replace(0, 1)

df[["tenure", "TotalCharges", "AvgMonthlySpend"]].head()

,tenure,TotalCharges,AvgMonthlySpend
0,1,29.85,29.850000
1,34,1889.50,55.573529
2,2,108.15,54.075000
3,45,1840.75,40.905556
4,2,151.65,75.825000


In [21]:
service_columns = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]

df["TotalServices"] = (
    df[service_columns]
    .apply(lambda row: (row == "Yes").sum(), axis=1)
)

df[["TotalServices"] + service_columns].head()

,TotalServices,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,1,No,Yes,No,No,No,No
1,2,Yes,No,Yes,No,No,No
2,2,Yes,Yes,No,No,No,No
3,3,Yes,No,Yes,Yes,No,No
4,0,No,No,No,No,No,No


In [22]:
print("New features created:")
print("1. AvgMonthlySpend")
print("2. TotalServices")

print("\nNew shape:", df.shape)

df[["AvgMonthlySpend", "TotalServices"]].describe()

New features created:
1. AvgMonthlySpend
2. TotalServices

New shape: (7043, 23)


,AvgMonthlySpend,TotalServices
count,7043.000000,7043.000000
mean,66.880842,2.037910
std,60.660378,1.847682
min,13.775000,0.000000
25%,36.255000,0.000000
50%,70.450000,2.000000
75%,90.285826,3.000000
max,1397.475000,6.000000


In [23]:
df = df.drop("customerID", axis=1)

print("Shape after removing customerID:", df.shape)

Shape after removing customerID: (7043, 22)


In [24]:
categorical_columns = df.select_dtypes(
    include=["object"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'Churn']


In [25]:
df = pd.get_dummies(
    df,
    columns=categorical_columns,
    drop_first=True,
    dtype=int
)

print("Shape after encoding:", df.shape)

Shape after encoding: (7043, 33)


In [26]:
print(df.head())
print("\nData types:")
print(df.dtypes.value_counts())

   SeniorCitizen  tenure  MonthlyCharges  TotalCharges  AvgMonthlySpend  \
0              0       1           29.85         29.85        29.850000   
1              0      34           56.95       1889.50        55.573529   
2              0       2           53.85        108.15        54.075000   
3              0      45           42.30       1840.75        40.905556   
4              0       2           70.70        151.65        75.825000   

   TotalServices  gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0              1            0            1               0                 0   
1              2            1            0               0                 1   
2              2            1            0               0                 1   
3              3            1            0               0                 0   
4              0            0            0               0                 1   

   ...  StreamingTV_Yes  StreamingMovies_No internet service  \
0  .

In [27]:
numerical_columns = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
    "AvgMonthlySpend"
]

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[
        (df[column] < lower_bound) |
        (df[column] > upper_bound)
    ]

    print(f"{column}: {len(outliers)} outliers")

tenure: 0 outliers
MonthlyCharges: 0 outliers
TotalCharges: 0 outliers
AvgMonthlySpend: 11 outliers


In [28]:
before_outlier_removal = df.shape[0]

for column in numerical_columns:
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    df = df[
        (df[column] >= lower_bound) &
        (df[column] <= upper_bound)
    ]

after_outlier_removal = df.shape[0]

print("Rows before outlier removal:", before_outlier_removal)
print("Rows after outlier removal:", after_outlier_removal)
print("Rows removed:", before_outlier_removal - after_outlier_removal)

Rows before outlier removal: 7043
Rows after outlier removal: 7032
Rows removed: 11


In [29]:
print("Shape after outlier removal:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Shape after outlier removal: (7032, 33)
Missing values: 0
Duplicate rows: 22


In [30]:
duplicates_before = df.duplicated().sum()

df = df.drop_duplicates().reset_index(drop=True)

print("Duplicate rows removed:", duplicates_before)
print("Shape after removing duplicates:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())

Duplicate rows removed: 22
Shape after removing duplicates: (7010, 33)
Remaining duplicates: 0


In [31]:
from sklearn.preprocessing import StandardScaler

In [32]:
scaler = StandardScaler()

df[numerical_columns] = scaler.fit_transform(
    df[numerical_columns]
)

In [33]:
print("Scaled numerical features:")

print(df[numerical_columns].describe().T)

Scaled numerical features:
                  count          mean       std       min       25%       50%  \
tenure           7010.0 -1.125110e-16  1.000071 -1.285566 -0.959284 -0.143580   
MonthlyCharges   7010.0 -5.676233e-17  1.000071 -1.551384 -0.969266  0.183328   
TotalCharges     7010.0 -4.003771e-17  1.000071 -1.002159 -0.830315 -0.391095   
AvgMonthlySpend  7010.0 -8.108904e-18  1.000071 -1.694627 -0.942540  0.185950   

                      75%       max  
tenure           0.957620  1.610184  
MonthlyCharges   0.831974  1.791638  
TotalCharges     0.669481  2.821089  
AvgMonthlySpend  0.840988  1.873497  


In [34]:
print("Final shape:", df.shape)
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

Final shape: (7010, 33)
Missing values: 0
Duplicate rows: 0


In [35]:
print("========== BEFORE vs AFTER PREPROCESSING ==========")

print("\nBEFORE PREPROCESSING")
print("Rows:", 7043)
print("Columns:", 21)

print("\nAFTER PREPROCESSING")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

========== BEFORE vs AFTER PREPROCESSING ==========

BEFORE PREPROCESSING
Rows: 7043
Columns: 21

AFTER PREPROCESSING
Rows: 7010
Columns: 33
Missing values: 0
Duplicate rows: 0


In [36]:
print("Engineered Features:")
print("- AvgMonthlySpend")
print("- TotalServices")

Engineered Features:
- AvgMonthlySpend
- TotalServices


In [37]:
df.head()

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,AvgMonthlySpend,TotalServices,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,-1.285566,-1.165523,-0.997284,-1.161688,1,0,1,0,0,...,0,0,0,0,0,1,0,1,0,0
1,0,0.060346,-0.264071,-0.176848,-0.308868,2,1,0,0,1,...,0,0,0,1,0,0,0,0,1,0
2,0,-1.244781,-0.367189,-0.962740,-0.358549,2,1,0,0,1,...,0,0,0,0,0,1,0,0,1,1
3,0,0.508983,-0.751387,-0.198355,-0.795160,3,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,-1.244781,0.193308,-0.943549,0.362535,0,0,0,0,1,...,0,0,0,0,0,1,0,1,0,1


In [38]:
print("Non-numeric columns:")
print(df.select_dtypes(exclude=["number"]).columns.tolist())

Non-numeric columns:
[]


In [39]:
df.to_csv("telco_customer_churn_cleaned.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [40]:
import os

print(os.path.exists("telco_customer_churn_cleaned.csv"))

True


In [41]:
print("========== FINAL PREPROCESSING SUMMARY ==========")

print("Original dataset:")
print("Rows: 7043")
print("Columns: 21")

print("\nFinal cleaned dataset:")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nData quality:")
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

print("\nEngineered features:")
print("1. AvgMonthlySpend")
print("2. TotalServices")

print("\nPreprocessing operations:")
print("✓ Missing/invalid values handled")
print("✓ Categorical variables encoded")
print("✓ Outliers removed")
print("✓ Duplicate rows removed")
print("✓ Numerical features standardized")

========== FINAL PREPROCESSING SUMMARY ==========
Original dataset:
Rows: 7043
Columns: 21

Final cleaned dataset:
Rows: 7010
Columns: 33

Data quality:
Missing values: 0
Duplicate rows: 0

Engineered features:
1. AvgMonthlySpend
2. TotalServices

Preprocessing operations:
✓ Missing/invalid values handled
✓ Categorical variables encoded
✓ Outliers removed
✓ Duplicate rows removed
✓ Numerical features standardized


In [42]:
df.head(10)

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,AvgMonthlySpend,TotalServices,gender_Male,Partner_Yes,Dependents_Yes,PhoneService_Yes,...,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,Churn_Yes
0,0,-1.285566,-1.165523,-0.997284,-1.161688,1,0,1,0,0,...,0,0,0,0,0,1,0,1,0,0
1,0,0.060346,-0.264071,-0.176848,-0.308868,2,1,0,0,1,...,0,0,0,1,0,0,0,0,1,0
2,0,-1.244781,-0.367189,-0.962740,-0.358549,2,1,0,0,1,...,0,0,0,0,0,1,0,0,1,1
3,0,0.508983,-0.751387,-0.198355,-0.795160,3,1,0,0,0,...,0,0,0,1,0,0,0,0,0,0
4,0,-1.244781,0.193308,-0.943549,0.362535,0,0,0,0,1,...,0,0,0,0,0,1,0,1,0,1
5,0,-1.000070,1.156297,-0.648467,1.248971,3,0,0,0,1,...,1,0,1,0,0,1,0,1,0,1
6,0,-0.429077,0.805363,-0.150421,0.786370,2,1,0,1,1,...,1,0,0,0,0,1,1,0,0,0
7,0,-0.918499,-1.168849,-0.877262,-1.150416,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
8,0,-0.184365,1.327606,0.333397,1.455351,4,0,1,0,1,...,1,0,1,0,0,1,0,1,0,1
9,0,1.202332,-0.290682,0.528353,-0.286200,2,1,0,1,1,...,0,0,0,1,0,0,0,0,0,0


In [43]:
print("Final dataset shape:", df.shape)

Final dataset shape: (7010, 33)


### Key Findings

* Original dataset: 7,043 rows × 21 columns.
* Handled missing/invalid values and converted `TotalCharges` to numeric.
* Created 2 new features: `AvgMonthlySpend` and `TotalServices`.
* Encoded categorical variables and removed outliers.
* Scaled numerical features using StandardScaler.
* Final dataset: 7,010 rows × 33 columns, with 0 missing values and 0 duplicates.


In [44]:
from google.colab import files

files.download("telco_customer_churn_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>